# Paired audio-video fusion

RAVDESS emotion demo. Livingstone & Russo (2018), DOI 10.1371/journal.pone.0196391; CC BY-NC-SA 4.0. Label tidak dikonversi menjadi fatigue.

In [ ]:
from pathlib import Path
import os
for key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMBA_NUM_THREADS'):
    os.environ.setdefault(key, '4')
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
os.chdir(ROOT)
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from atc_fatigue.utils.runtime import load_config
config = load_config()
%matplotlib inline


In [ ]:
features = pd.read_parquet('data/interim/ravdess_features.parquet')
result = json.loads(Path('reports/metrics/ravdess_emotion_seed42_v1/metrics.json').read_text())
assert features.paired_recording_verified.all()
print(len(features), 'AV recordings;', features.subject_id.nunique(), 'actors')
display(pd.DataFrame({name: {k: scores['test'][k] for k in ['macro_f1', 'balanced_accuracy', 'accuracy']} for name, scores in result['models'].items()}).T)
display(result['splits'])

In [ ]:
split = pd.read_csv('reports/metrics/ravdess_emotion_seed42_v1/split_manifest.csv', dtype={'subject_id': str})
assert split.groupby('subject_id').split.nunique().max() == 1
from atc_fatigue.models.baselines import predict_artifact
holdout = split[split.split.eq('test')][['sample_id']].merge(features, on='sample_id')
pred = predict_artifact('models/ravdess_emotion_seed42_v1/audio_visual.joblib', holdout)
display(pd.DataFrame({'sample_id': holdout.sample_id, 'emotion_truth': holdout.target_value, 'emotion_prediction': pred}).head())

In [ ]:
display(pd.DataFrame({name: {k: scores[k] for k in ['macro_f1', 'balanced_accuracy']} for name, scores in result['models']['audio_visual']['missing_modality_test'].items()}).T)

Semua modalitas berasal dari recording AV yang sama. Fitur visual adalah sampled face appearance pada 2 FPS; tidak mengukur blink. Subset 8 aktor kecil, hasil tidak stabil, dan inference ablation tidak membuktikan missing-modality robustness. Split ini kebetulan berisi aktor male pada train dan female pada validation/test menurut metadata resmi; demographic shift membatasi interpretasi. Split tidak diubah setelah evaluasi. Berikutnya diperlukan paired ATC data dengan label fatigue yang tervalidasi.